In [3]:
# STEP 1: IMPORTING LIBRARIES

import pandas as pd
import numpy as np


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
# STEP 2: LOADING DATASET

df = pd.read_csv(r"D:\AI ML\DiabetesKaggle.csv")

print("Dataset Shape:", df.shape)

print("\nFirst 5 Rows:\n")
print(df.head())

Dataset Shape: (100000, 9)

First 5 Rows:

   gender   age  hypertension  heart_disease smoking_history    bmi  \
0  Female  80.0             0              1           never  25.19   
1  Female  54.0             0              0         No Info  27.32   
2    Male  28.0             0              0           never  27.32   
3  Female  36.0             0              0         current  23.45   
4    Male  76.0             1              1         current  20.14   

   HbA1c_level  blood_glucose_level  Outcome  
0          6.6                  140        0  
1          6.6                   80        0  
2          5.7                  158        0  
3          5.0                  155        0  
4          4.8                  155        0  


In [5]:
# STEP 3: HANDLING MISSING VALUES

print("Missing Values Before Handling:\n")
print(df.isnull().sum())

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == "object":
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

print("\nMissing Values After Handling:\n")
print(df.isnull().sum())

Missing Values Before Handling:

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
Outcome                0
dtype: int64

Missing Values After Handling:

gender                 0
age                    0
hypertension           0
heart_disease          0
smoking_history        0
bmi                    0
HbA1c_level            0
blood_glucose_level    0
Outcome                0
dtype: int64


In [6]:
# STEP 4: REMOVING OUTLIERS

outlier_cols = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]

rows_before = df.shape[0]

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

rows_after = df.shape[0]

print("Rows before:", rows_before)
print("Rows after :", rows_after)
print("Outliers removed:", rows_before - rows_after)

df.reset_index(drop=True, inplace=True)

Rows before: 100000
Rows after : 90387
Outliers removed: 9613


In [7]:
# STEP 5: ENCODING CATEGORICAL COLUMNS

df = pd.get_dummies(df, columns=["gender", "smoking_history"], drop_first=True)

print("Dataset Shape After Encoding:", df.shape)

print("\nColumns:\n")
print(df.columns)

Dataset Shape After Encoding: (90387, 14)

Columns:

Index(['age', 'hypertension', 'heart_disease', 'bmi', 'HbA1c_level',
       'blood_glucose_level', 'Outcome', 'gender_Male', 'gender_Other',
       'smoking_history_current', 'smoking_history_ever',
       'smoking_history_former', 'smoking_history_never',
       'smoking_history_not current'],
      dtype='object')


In [8]:
# STEP 6: SELECTING FINAL FEATURES AND TARGET

X_selected_raw = df[["HbA1c_level", "blood_glucose_level", "bmi", "age"]]
y = df["Outcome"]

print("Selected Features:\n")
print(X_selected_raw.head())

print("\nTarget Values:\n")
print(y.head())

Selected Features:

   HbA1c_level  blood_glucose_level    bmi   age
0          6.6                  140  25.19  80.0
1          6.6                   80  27.32  54.0
2          5.7                  158  27.32  28.0
3          5.0                  155  23.45  36.0
4          4.8                  155  20.14  76.0

Target Values:

0    0
1    0
2    0
3    0
4    0
Name: Outcome, dtype: int64


In [9]:
# STEP 7: USING TRAIN-TEST SPLIT

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_selected_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train_raw Shape:", X_train_raw.shape)
print("X_test_raw Shape :", X_test_raw.shape)
print("y_train Shape    :", y_train.shape)
print("y_test Shape     :", y_test.shape)

X_train_raw Shape: (72309, 4)
X_test_raw Shape : (18078, 4)
y_train Shape    : (72309,)
y_test Shape     : (18078,)


In [10]:
# STEP 8: SCALING THE SELECTED FEATURES

scaler_selected = StandardScaler()

X_train = scaler_selected.fit_transform(X_train_raw)
X_test = scaler_selected.transform(X_test_raw)

X_train = pd.DataFrame(
    X_train,
    columns=X_selected_raw.columns,
    index=X_train_raw.index
)

X_test = pd.DataFrame(
    X_test,
    columns=X_selected_raw.columns,
    index=X_test_raw.index
)

print("Scaled Training Data:\n")
print(X_train.head())

Scaled Training Data:

       HbA1c_level  blood_glucose_level       bmi       age
33208    -1.966363            -1.406429  0.564002  0.914608
59890     0.547245             0.578815 -0.439224  1.314222
89969    -0.458198            -0.130201  2.346373  0.248584
67358    -0.960920             0.578815 -1.306767 -0.461841
7267      0.647789            -0.243643  0.204239  0.825805


In [14]:
# STEP 9: SAVING FIRST 500 ROWS OF PREPROCESSED DATA

preprocessed_df = X_selected_raw.copy()
preprocessed_df["Outcome"] = y

preprocessed_500 = preprocessed_df.head(500)

preprocessed_500.to_csv(r"D:\AI ML\preprocessed_diabetes_500_data.csv", index=False)

print("First 500 rows of preprocessed data saved successfully")
print("File saved as: D:\\AI ML\\preprocessed_diabetes_500_data.csv")
print("Shape of saved data:", preprocessed_500.shape)

print("\nFirst 5 rows:\n")
print(preprocessed_500.head())

First 500 rows of preprocessed data saved successfully
File saved as: D:\AI ML\preprocessed_diabetes_500_data.csv
Shape of saved data: (500, 5)

First 5 rows:

   HbA1c_level  blood_glucose_level    bmi   age  Outcome
0          6.6                  140  25.19  80.0        0
1          6.6                   80  27.32  54.0        0
2          5.7                  158  27.32  28.0        0
3          5.0                  155  23.45  36.0        0
4          4.8                  155  20.14  76.0        0
